# Gate 1-2 — MBHT Setup + Reproduce A0 (RetailRocket)

**Mục đích:** chuẩn bị môi trường Colab, tải processed dataset chính thức của MBHT lên Google Drive, in schema (đặc biệt là mapping `item_type_list`), và reproduce baseline A0 (MBHT gốc, chưa sửa gì) trên **RetailRocket** — dataset nhỏ nhất, dùng để tiết kiệm compute unit trước khi mở rộng sang Tmall/IJCAI.

**Gate liên quan (theo file kế hoạch):**
- **G0 (Data):** processed data load được; behavior mapping đủ rõ để dựng transition. Cell 6 in ra `field2token_id["item_type_list"]` — đây là bằng chứng thực tế cho mapping, không giả định ID 0/1/2.
- **G1 (A0):** MBHT reproduce ổn định, metric hợp lý, pipeline deterministic đủ (≥2 run ổn định trước khi mở seed set đầy đủ).

**Quy tắc hạ tầng đã áp dụng trong notebook này:**
- Data + checkpoint lưu trên Google Drive (`/content/drive/MyDrive/DeAnThS/...`), KHÔNG lưu ở `/content` (mất khi disconnect).
- Có resume: nếu tìm thấy checkpoint cũ trong thư mục Drive tương ứng, sẽ tự động resume thay vì train lại từ đầu.
- Bắt đầu bằng RetailRocket (`retail_beh`) theo đúng yêu cầu tiết kiệm compute.
- Code clone trực tiếp từ repo gốc `yuh-yang/MBHT-KDD22`, pin đúng commit đã audit trên VPS: `4a2f3cfeaa3225a003b511111fbdeec38f04a1f9`. Đây là bước A0 (chưa có B1/B2), nên chưa cần pull từ fork riêng. Khi sang Gate B1/B2, notebook kế tiếp sẽ clone từ repo GitHub riêng của bạn (cần xác nhận URL fork lúc đó).

**Việc bạn cần làm:** chạy tuần tự các cell từ trên xuống. Nếu cell nào lỗi, dừng lại và dán nguyên văn output/lỗi lại cho tôi — đừng tự sửa để tránh lệch protocol.

## 1. Kiểm tra GPU + mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2. Định nghĩa đường dẫn cố định trên Drive

Mọi thứ cần giữ qua nhiều session (data, checkpoint, log, config đã khóa) đều nằm dưới `PROJECT_ROOT` trên Drive. `/content/...` chỉ chứa code (ephemeral, clone lại mỗi lần).

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/DeAnThS'
DATA_DIR = os.path.join(PROJECT_ROOT, 'dataset')          # processed datasets (persist)
CKPT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')      # RecBole checkpoint_dir (persist)
LOG_DIR = os.path.join(PROJECT_ROOT, 'logs')              # training logs (persist)
RUN_META_DIR = os.path.join(PROJECT_ROOT, 'run_meta')     # frozen config/commit records (persist)
CODE_DIR = '/content/MBHT-KDD22'                          # code (ephemeral, re-cloned each session)

for d in [DATA_DIR, CKPT_DIR, LOG_DIR, RUN_META_DIR]:
    os.makedirs(d, exist_ok=True)

# Fork rieng cua du an (co patch env-compat + sau nay se co B1/B2), KHONG con la upstream goc.
REPO_URL = 'https://github.com/nguyenlmhcm/MBHT-KDD22.git'
BRANCH = 'bt-mbht'
# Commit da audit + patch device-mismatch bug tren VPS (Gate 1). Cap nhat tay moi khi VPS push commit moi can dung.
PINNED_COMMIT = '6066aed7189bc0295b95fbc26232b57dcdad78ef'  # bt-mbht HEAD sau patch torch.load weights_only (Gate 1)

print('DATA_DIR     :', DATA_DIR)
print('CKPT_DIR     :', CKPT_DIR)
print('LOG_DIR      :', LOG_DIR)
print('RUN_META_DIR :', RUN_META_DIR)
print('CODE_DIR     :', CODE_DIR)

## 3. Clone repo (pinned commit) + cài dependency

In [ ]:
import subprocess, time

# The dependency cell below does `%cd {CODE_DIR}`, so re-running this cell
# would delete the directory the process is standing in -- after which every
# git call fails with "Unable to read current working directory". Step out
# first so a re-run is always safe.
os.chdir('/content')

if os.path.isdir(CODE_DIR):
    print('CODE_DIR already exists, removing for a clean clone...')
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)

# Colab dung chung dai IP voi nhieu user khac -> doi luc bi GitHub rate-limit
# anonymous clone thoang qua (exit 128). Retry vai lan truoc khi bao loi that.
MAX_CLONE_RETRIES = 5
for attempt in range(1, MAX_CLONE_RETRIES + 1):
    result = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, CODE_DIR], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'Clone OK (attempt {attempt}/{MAX_CLONE_RETRIES})')
        break
    print(f'Clone attempt {attempt}/{MAX_CLONE_RETRIES} failed (exit {result.returncode}):')
    print(result.stderr.strip())
    if os.path.isdir(CODE_DIR):
        subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    if attempt == MAX_CLONE_RETRIES:
        raise RuntimeError(f'git clone failed after {MAX_CLONE_RETRIES} attempts. Last stderr above.')
    wait_s = 10 * attempt
    print(f'Retrying in {wait_s}s...')
    time.sleep(wait_s)

subprocess.run(['git', '-C', CODE_DIR, 'checkout', PINNED_COMMIT], check=True)

actual_commit = subprocess.run(
    ['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=True
).stdout.strip()
assert actual_commit == PINNED_COMMIT, f'Commit mismatch: {actual_commit} != {PINNED_COMMIT}'
print('Checked out commit:', actual_commit)


In [ ]:
%cd {CODE_DIR}
# torch is preinstalled on Colab; only install the remaining requirements.
!pip install -q hyperopt pandas tqdm scikit_learn pyyaml colorlog colorama tensorboard
!pip install -q gdown

## 4. Tải processed dataset chính thức (Google Drive của tác giả) — chỉ 1 lần, lưu trên Drive

Nguồn: README chính thức của MBHT-KDD22, file zip chứa cả 3 dataset (Tmall/IJCAI/RetailRocket) đã qua xử lý.
Nếu `DATA_DIR` đã có sẵn nội dung hợp lệ (đã tải từ lần chạy trước), cell này sẽ bỏ qua bước tải lại.

In [ ]:
OFFICIAL_GDRIVE_FILE_ID = '1OFT_5Xp_az-GSHIl7QEPB9zhulbooLzE'
zip_path = os.path.join(PROJECT_ROOT, 'mbht_official_datasets.zip')

already_have_data = len(os.listdir(DATA_DIR)) > 0
if already_have_data:
    print('DATA_DIR is not empty, skipping download. Contents:')
    print(os.listdir(DATA_DIR))
else:
    print('Downloading official processed datasets zip (one-time)...')
    import gdown
    gdown.download(id=OFFICIAL_GDRIVE_FILE_ID, output=zip_path, quiet=False)
    print('Unzipping into DATA_DIR (persisted on Drive)...')
    subprocess.run(['unzip', '-q', zip_path, '-d', DATA_DIR], check=True)
    print('Done. DATA_DIR contents:')
    print(os.listdir(DATA_DIR))

In [ ]:
# CAU TRUC THUC TE DA XAC NHAN (khong con la gia dinh):
#   DATA_DIR/MBHT_dataset/<retail_beh|tmall_beh|ijcai_beh>/<name>.{train,test}.inter
# => RecBole 'data_path' phai tro vao DATA_DIR/MBHT_dataset, khong phai DATA_DIR truc tiep.
DATASET_ROOT = os.path.join(DATA_DIR, 'MBHT_dataset')
assert os.path.isdir(DATASET_ROOT), f'Khong tim thay {DATASET_ROOT}. Kiem tra lai buoc tai/giai nen o cell truoc.'

for root, dirs, files in os.walk(DATASET_ROOT):
    depth = root.replace(DATASET_ROOT, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or DATASET_ROOT}/')
    if depth == 1:
        for f in files:
            print(f'{indent}  {f}')

print()
print('DATASET_ROOT (dung cho RecBole data_path):', DATASET_ROOT)


## 5. Load dataset qua RecBole + in schema (RetailRocket)

Đây là bằng chứng thực tế cho `field2token_id["item_type_list"]` — bắt buộc phải xem trước khi B1/B2 giả định bất kỳ ID nào cho behavior type. Dùng `DATASET_ROOT` (đã xác nhận ở cell trên = `DATA_DIR/MBHT_dataset`) làm `data_path`, không dùng `DATA_DIR` trực tiếp — zip giải nén ra thêm một cấp thư mục cha `MBHT_dataset`. Nếu tên dataset con khác `retail_beh` (xem output cell trên), sửa biến `DATASET_NAME` cho khớp.

In [ ]:
DATASET_NAME = 'retail_beh'  # sửa nếu tên thư mục thực tế khác (xem output cell 4)

from recbole.config import Config
from recbole.data import create_dataset

schema_config_dict = {
    'data_path': DATASET_ROOT,
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'MAX_ITEM_LIST_LENGTH': 200,
}

schema_config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=schema_config_dict)
dataset = create_dataset(schema_config)
print(dataset)
print()
print('=== field2token_id[item_type_list] (behavior vocabulary — GROUND TRUTH, khong doan) ===')
print(dataset.field2token_id['item_type_list'])
print()
print('=== field2id_token[item_type_list] (id -> token nguoc lai) ===')
print(dataset.field2id_token['item_type_list'])
print()
print('n_users (sessions):', dataset.user_num)
print('n_items           :', dataset.item_num)

In [ ]:
# Vai sequence mau + thong ke do dai, de doi chieu voi gia dinh trong plan (VIEW/CART/BUY,...)
import numpy as np

inter_feat = dataset.inter_feat
item_lists = inter_feat['item_id_list']
type_lists = inter_feat['item_type_list']

lengths = [int((row != 0).sum()) for row in item_lists[:2000]]
print('Seq length stats (sample 2000) -> min/mean/median/max:',
      min(lengths), float(np.mean(lengths)), float(np.median(lengths)), max(lengths))

print()
print('=== 3 sample (item_id_list, item_type_list) pairs, non-zero part only ===')
for i in range(3):
    items = item_lists[i]
    types = type_lists[i]
    mask = items != 0
    print(f'sample {i}: items={items[mask].tolist()}')
    print(f'          types={types[mask].tolist()}')

## 6. Khóa (freeze) commit + config vào Drive — dùng chung cho A0/A1/A2/A3

Ghi ra file JSON trong `RUN_META_DIR` để mọi run sau này (baseline lẫn B1/B2) tham chiếu cùng một config, tránh lệch protocol giữa các experiment.

In [ ]:
import json

# Config nay copy dung tu run_MBHT.py + nhanh retail_beh cua repo goc (KHONG doi).
frozen_config = {
    'commit': PINNED_COMMIT,
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'patches': [
        '71f078a: fix device-mismatch in build_Gs_unique (env-compat, no logic change)',
        'acdc9e7: fix np.float/np.bool deprecated numpy aliases (env-compat, no logic change)',
        '6066aed: fix torch.load weights_only default flip in trainer.py (env-compat, no logic change)',
    ],
    'dataset': DATASET_NAME,
    'USER_ID_FIELD': 'session_id',
    'load_col': None,
    'neg_sampling': None,
    'benchmark_filename': ['train', 'test'],
    'alias_of_item_id': ['item_id_list'],
    'topk': [5, 10, 101],
    'metrics': ['Recall', 'NDCG', 'MRR'],
    'valid_metric': 'NDCG@10',
    'eval_args': {'mode': 'full', 'order': 'TO'},
    'MAX_ITEM_LIST_LENGTH': 200,
    'train_batch_size': 64,
    'eval_batch_size': 128,
    'hyper_len': 6,
    'scales': [5, 4, 20],
    'enable_hg': 1,
    'enable_ms': 1,
    'customized_eval': 1,
}

meta_path = os.path.join(RUN_META_DIR, f'A0_frozen_config_{DATASET_NAME}.json')
with open(meta_path, 'w') as f:
    json.dump(frozen_config, f, indent=2, ensure_ascii=False)
print('Frozen config saved to:', meta_path)
print(json.dumps(frozen_config, indent=2, ensure_ascii=False))

## 7. Reproduce A0 (MBHT goc, RetailRocket) — voi checkpoint/resume tren Drive

**QUYET DINH PROTOCOL (da duoc duyet, khong con la open question):** giu nguyen protocol goc cua `run_MBHT.py` — **KHONG** dung `--validation`. Nghia la `test_data` duoc truyen thang vao vi tri `valid_data` cua `trainer.fit()`, early-stopping/model-selection chon theo test-set score, khong co validation set doc lap. Day la protocol GOC cua tac gia, giu de so sanh truc tiep voi so published.

**Ly do gain van hop le:** vi A0, A1, A2, A3 dung CUNG mot protocol nay, leakage (neu co) trieu tieu khi lay hieu so giua cac variant. Xem file ke hoach muc 8.2 va bao cao Gate 0.

**RANG BUOC BAT BUOC cho moi notebook A1/A2/A3 sau nay (khong duoc lech):**
- Cung `epochs` toi da, cung `stopping_step` (early-stopping patience).
- Cung tieu chi chon best checkpoint (`valid_metric`, hien tai la `NDCG@10`).
- Cung seed set, dung y het gia tri seed cho tung lan chay tuong ung giua cac variant.
- Khong doi bat ky config nao khac ngoai chinh module B1/B2 dang duoc bat/tat.

**Ghi chu cho paper (khong sua code):** trong phan Experimental Setup se ghi ro la theo dung protocol danh gia goc cua MBHT de dam bao so sanh nhat quan voi so published, va moi bien the A0-A3 dung chung protocol nay.

Cell nay se **tu dong resume** neu tim thay checkpoint cu cung `RUN_TAG` trong `CKPT_DIR`.

In [ ]:
import glob
from logging import getLogger
from recbole.data.utils import get_dataloader, create_samplers
from recbole.model.sequential_recommender.mbht import MBHT
from recbole.utils import init_logger, init_seed, get_trainer, set_color

SEED = 2020        # doi gia tri nay cho tung seed trong seed set (khong cherry-pick)
RUN_TAG = f'A0-MBHT-{DATASET_NAME}-seed{SEED}'

run_ckpt_dir = os.path.join(CKPT_DIR, RUN_TAG)
run_log_dir = os.path.join(LOG_DIR, RUN_TAG)
os.makedirs(run_ckpt_dir, exist_ok=True)
os.makedirs(run_log_dir, exist_ok=True)

config_dict = dict(frozen_config)
config_dict.pop('commit'); config_dict.pop('repo_url'); config_dict.pop('dataset')
config_dict.update({
    'data_path': DATASET_ROOT,
    'checkpoint_dir': run_ckpt_dir,
    'gpu_id': 0,
    'seed': SEED,
    'abaltion': '',
})

config = Config(model='MBHT', dataset=DATASET_NAME, config_dict=config_dict)
init_seed(config['seed'], config['reproducibility'])
init_logger(config, log_root=run_log_dir)
logger = getLogger()
logger.info(f'RUN_TAG={RUN_TAG}  commit={PINNED_COMMIT}')
logger.info(config)

train_dataset, test_dataset = dataset.build() if 'dataset' in globals() else create_dataset(config).build()
train_sampler, test_sampler = create_samplers(config, dataset, [train_dataset, test_dataset])
train_data = get_dataloader(config, 'train')(config, train_dataset, train_sampler, shuffle=True)
test_data = get_dataloader(config, 'test')(config, test_dataset, test_sampler, shuffle=False)

model = MBHT(config, train_data.dataset).to(config['device'])
logger.info(model)
trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

existing_ckpts = sorted(glob.glob(os.path.join(run_ckpt_dir, '*.pth')), key=os.path.getmtime)
if existing_ckpts:
    resume_file = existing_ckpts[-1]
    print(f'Found existing checkpoint, resuming from: {resume_file}')
    trainer.resume_checkpoint(resume_file)
else:
    print('No existing checkpoint found, starting fresh.')

test_score, test_result = trainer.fit(
    train_data, test_data, saved=True, show_progress=config['show_progress']
)
print(set_color('test result', 'yellow') + f': {test_result}')

## 8. Ket qua can dan lai cho Claude (VPS)

Sau khi chay xong (hoac neu loi), dan lai nguyen van:
1. Output cell 4 (cay thu muc dataset thuc te).
2. Output cell 5 (`field2token_id['item_type_list']`, `field2id_token`, n_users, n_items).
3. Output cell 5b (sample sequences + length stats).
4. Dong log cuoi cung + `test_result` cua cell 7.
5. Duong dan file log day du trong `LOG_DIR` (de doi chieu sau).
6. Bat ky loi/traceback nao gap phai, dan nguyen van, khong tom tat.